# Projekt Datenanalyse – Consumer Complaint Dataset

## 1. Projektziel und Datensatz
Ziel des Projekts ist es, aus einer großen Menge unstrukturierter Verbraucherbeschwerden die am häufigsten vorkommenden Themen mithilfe von NLP-Techniken zu extrahieren.

Als Datengrundlage wird der Consumer Complaint Dataset des Consumer Financial Protection Bureau (CFPB) verwendet. Der Datensatz enthält schriftliche Verbraucherbeschwerden sowie zusätzliche Metadaten und bereits vorgegebene Kategorien. Für die eigentliche Themenanalyse wird ein zufällig ausgewählter Teil des Datensatzes mit ausschließlich freien Beschwerdetexten verwendet, die vorgegebenen Kategorien werden für die Datenanalyse nicht verwendet.

## 2. Datenimport

In [4]:
import pandas as pd
# Datensatz wurde zur weiteren Verarbeitung anfangs eingelesen.
# df = pd.read_csv(r"C:\Users\goetz\Dropbox\Studium\5. Semester\Data Analyst Projekt\dataset\complaints.csv")

# Aufruf des mittlerweile gespeicherten Datesatzes mit der StiPo der 20000 Datensätze
sample = pd.read_csv(
    r"C:\Users\goetz\Dropbox\Studium\5. Semester\Data Analyst Projekt\dataset\complaints_sample_20000.csv"
)

## 3. Explorative Datenanalyse

In [6]:
# df.shape

(2023066, 11)

Der Datensatz enthält insgesamt 2.023.066 Einträge und 11 Merkmale. Da die Verarbeitung von über zwei Millionen Texten einen sehr hohen Rechenaufwand verursachen würde, soll für die weitere Analyse nur eine Stichprobe verwendet werden.

Wie groß diese Stichprobe sein soll, wird nach einer ersten Untersuchung der Daten entschieden. Dabei soll die Stichprobe groß genug sein, um häufig vorkommende Themen erkennen zu können, gleichzeitig aber auch mit den zur Verfügung stehenden Mitteln gut verarbeitet werden können.

In [9]:
# df.columns

**Enthaltene Merkmale:**

"Unnamed: 0", "product_5", "narrative", "Product", "Date received", "Sub-product", "Issue", 
"Sub-issue", "Company", "State", "Timely response?"

In [11]:
# df["narrative"].head()

Die Spalte narrative enthält die frei formulierten Beschwerdetexte und bildet damit die Grundlage für die weitere Textanalyse. Für die folgenden Schritte wird daher ausschließlich diese Textspalte betrachtet.

## 4. Auswahl der Textdaten

Da der vollständige Datensatz über zwei Millionen Beschwerdetexte enthält, wird für die weitere Analyse eine Stichprobe von 20.000 Texten verwendet. Dies entspricht ungefähr 1 % des gesamten Datensatzes. Die Stichprobe soll groß genug sein, um häufig vorkommende Themen erkennen zu können, gleichzeitig aber den Rechenaufwand bei der späteren Vektorisierung und Themenanalyse begrenzen. Die Auswahl erfolgt zufällig und mit einem festen `random_state`, damit bei einer erneuten Ausführung dieselben Datensätze ausgewählt werden.

Vor der Auswahl der Stichprobe werden leere Einträge sowie Texte mit weniger als zehn Wörtern ausgeschlossen. Sehr kurze Texte können beispielsweise nur aus einer Anrede oder wenigen allgemeinen Wörtern bestehen und enthalten damit kaum Informationen über den eigentlichen Inhalt der Beschwerde. Mit einer Mindestlänge von zehn Wörtern soll sichergestellt werden, dass die ausgewählten Texte zumindest einen gewissen inhaltlichen Umfang besitzen.

In [14]:
CREATE_SAMPLE = False

if CREATE_SAMPLE:

    df = pd.read_csv(
        r"C:\Users\goetz\Dropbox\Studium\5. Semester\Data Analyst Projekt\dataset\complaints.csv"
    )

    valid_narratives = df[
        df["narrative"].notna() &
        (df["narrative"].str.split().str.len() >= 10)
    ]
    sample = valid_narratives.sample(
        n=20000,
        random_state=42
    )

    sample.to_csv(
        r"C:\Users\goetz\Dropbox\Studium\5. Semester\Data Analyst Projekt\dataset\complaints_sample_20000.csv",
        index=False
    )

Die ausgewählte Stichprobe mit 20.000 Datensätzen wurde als eigene CSV-Datei gespeichert. Für die weiteren Analyseschritte wird diese Datei verwendet. Dadurch muss der vollständige Datensatz mit über zwei Millionen Einträgen bei späteren Sitzungen nicht erneut verarbeitet werden. Der obige Code wurde so angepasst, dass das Notebook weiterhin vollständig über Run All ausgeführt werden kann, ohne die Stichprobe erneut zu erstellen.

## 5. Textvorverarbeitung

Vor der Vektorisierung wurde geprüft, welche Schritte der Textvorverarbeitung für die Analyse notwendig sind. Dabei wurde festgestellt, dass ein großer Teil der üblichen NLP-Vorverarbeitung bereits durch die später verwendeten Vektorisierungsverfahren übernommen werden kann.

Eine separate Tokenisierung ist nicht notwendig, da "CountVectorizer" und "TfidfVectorizer" die Texte selbst in einzelne Wörter bzw. Tokens zerlegen. Beide Verfahren können außerdem die Texte in Kleinschreibung umwandeln und Satz- und Sonderzeichen bei der Bildung der Wörter weitgehend unberücksichtigt lassen. Auch englische Stoppwörter können direkt bei der Vektorisierung entfernt werden.

Auf eine zusätzliche Lemmatisierung wird zunächst verzichtet. Nach der Vektorisierung wird geprüft, ob unterschiedliche Wortformen häufig als einzelne Merkmale auftreten und dadurch die Ergebnisse beeinflussen. Sollte dies der Fall sein, kann die Lemmatisierung anschließend ergänzt werden.

Auch Zahlen bleiben zunächst erhalten. Nach der Vektorisierung wird geprüft, ob häufig vorkommende Zahlen unter den relevanten Merkmalen auftreten und die Themenanalyse beeinflussen. Falls dies der Fall ist, können diese anschließend entfernt werden.

Die Beschwerdetexte werden daher zunächst ohne zusätzliche manuelle Bereinigung an die Vektorisierungsverfahren übergeben.

## 6. Vektorisierung

Damit die Beschwerdetexte mit mathematischen Verfahren analysiert werden können, müssen die Texte zunächst in eine numerische Form umgewandelt werden. Dafür werden zwei unterschiedliche Verfahren verwendet und anschließend miteinander verglichen.

Als erstes Verfahren wird Bag-of-Words verwendet. Dabei wird für jedes Dokument erfasst, wie häufig die einzelnen Wörter vorkommen. Als zweites Verfahren wird TF-IDF eingesetzt. Hierbei wird zusätzlich berücksichtigt, wie häufig ein Wort im gesamten Datensatz vorkommt. Wörter, die in vielen Dokumenten vorkommen, erhalten dadurch eine geringere Gewichtung.

Bei beiden Verfahren werden die Texte in Kleinschreibung umgewandelt, tokenisiert und englische Stoppwörter entfernt.

### 6.1 Bag-of-Words

Der CountVectorizer übernimmt bereits einige Schritte der Textvorverarbeitung. Die Texte werden standardmäßig in Kleinbuchstaben umgewandelt und bei der Vektorisierung in einzelne Tokens zerlegt. Satz- und Sonderzeichen werden dabei weitgehend nicht als eigene Merkmale berücksichtigt. Zusätzlich werden mit der Einstellung "stop_words='english'" häufig vorkommende englische Stoppwörter entfernt.

Weitere Bereinigungsschritte werden zunächst nicht vorgenommen. Nach der Vektorisierung wird geprüft, ob sich unter den häufigsten Merkmalen weitere inhaltlich bedeutungslose Begriffe befinden, die für die spätere Analyse entfernt werden sollten.

In [22]:
from sklearn.feature_extraction.text import CountVectorizer

# Bag-of-Words
bow_vectorizer = CountVectorizer(stop_words="english")

bow_matrix = bow_vectorizer.fit_transform(sample["narrative"])

In [23]:
bow_matrix.shape

(20000, 22883)

In [24]:
# Häufigste Wörter im Bag-of-Words-Modell

import numpy as np

word_counts = np.asarray(bow_matrix.sum(axis=0)).flatten()
feature_names = bow_vectorizer.get_feature_names_out()

top_words = sorted(
    zip(feature_names, word_counts),
    key=lambda x: x[1],
    reverse=True
)[:30]

top_words

[('xxxx', 275417),
 ('xx', 48567),
 ('credit', 37275),
 ('account', 29466),
 ('information', 21335),
 ('report', 21253),
 ('consumer', 16135),
 ('reporting', 16014),
 ('00', 15507),
 ('15', 10273),
 ('accounts', 9605),
 ('payment', 8943),
 ('debt', 8295),
 ('section', 8242),
 ('bank', 7335),
 ('did', 7235),
 ('company', 6835),
 ('received', 6730),
 ('loan', 6583),
 ('card', 6473),
 ('balance', 6472),
 ('agency', 6177),
 ('time', 6160),
 ('number', 5989),
 ('told', 5978),
 ('sent', 5798),
 ('days', 5477),
 ('date', 5382),
 ('states', 5360),
 ('letter', 5265)]

Die Betrachtung der häufigsten Merkmale zeigt, dass die Anonymisierungsplatzhalter "xxxx" und "xx" sehr häufig vorkommen und dadurch die spätere Themenanalyse beeinflussen könnten. Da diese keine inhaltliche Bedeutung besitzen, werden sie entfernt.

Außerdem treten reine Zahlen bereits unter den häufigsten Merkmalen auf. Da einzelne Zahlen für die gesuchten Themen ebenfalls keinen wesentlichen Inhalt liefern, werden auch diese aus der weiteren Analyse ausgeschlossen.

Unterschiedliche Wortformen wie "account" und "accounts" bleiben zunächst erhalten. Ob hierfür eine Lemmatisierung notwendig ist, wird im weiteren Verlauf geprüft.

In [26]:
# Bag-of-Words - bereinigter Durchlauf

bow_vectorizer_clean = CountVectorizer(
    stop_words="english",
    token_pattern=r"(?u)\b(?!x+\b)[a-zA-Z]{2,}\b" # alle Zahlen und xx... Wörter ausschließen
)

bow_matrix_clean = bow_vectorizer_clean.fit_transform(sample["narrative"])

In [27]:
bow_matrix_clean.shape

(20000, 21184)

In [28]:
# Häufigste Wörter nach der Bereinigung

word_counts_clean = np.asarray(bow_matrix_clean.sum(axis=0)).flatten()
feature_names_clean = bow_vectorizer_clean.get_feature_names_out()

top_words_clean = sorted(
    zip(feature_names_clean, word_counts_clean),
    key=lambda x: x[1],
    reverse=True
)[:30]

top_words_clean

[('credit', 37275),
 ('account', 29466),
 ('information', 21335),
 ('report', 21253),
 ('consumer', 16135),
 ('reporting', 16014),
 ('accounts', 9605),
 ('payment', 8943),
 ('debt', 8295),
 ('section', 8242),
 ('bank', 7335),
 ('did', 7235),
 ('company', 6835),
 ('received', 6730),
 ('loan', 6583),
 ('card', 6473),
 ('balance', 6472),
 ('agency', 6177),
 ('time', 6160),
 ('number', 5989),
 ('told', 5978),
 ('sent', 5798),
 ('days', 5477),
 ('date', 5382),
 ('states', 5360),
 ('letter', 5265),
 ('payments', 5240),
 ('inaccurate', 4952),
 ('late', 4909),
 ('called', 4897)]

Nach der Bereinigung enthalten die 20.000 Texte noch 21.184 unterschiedliche Merkmale. Die zuvor häufig vorkommenden Anonymisierungszeichen und Zahlen treten unter den häufigsten Begriffen nicht mehr auf.

### 6.2 TF-IDF
Auch der TfidfVectorizer übernimmt bei der Vektorisierung bereits verschiedene Schritte der Textvorverarbeitung. Dazu gehören die Umwandlung in Kleinbuchstaben, die Tokenisierung und die weitgehende Nichtberücksichtigung von Satz- und Sonderzeichen. Englische Stoppwörter werden über die entsprechende Einstellung ebenfalls entfernt. Für Bag-of-Words und TF-IDF werden dabei die gleichen Einstellungen zur Textvorverarbeitung verwendet, damit die Ergebnisse der beiden Vektorisierungsverfahren möglichst gut miteinander vergleichbar sind.

Die bei der vorherigen Prüfung identifizierten bedeutungslosen Platzhalter und Zahlen werden ebenfalls bei beiden Verfahren ausgeschlossen.

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF - mit den gleichen Bereinigungseinstellungen wie Bag-of-Words

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    token_pattern=r"(?u)\b(?!x+\b)[a-zA-Z]{2,}\b"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(sample["narrative"])

In [32]:
tfidf_matrix.shape

(20000, 21184)

In [33]:
# Wörter mit den höchsten TF-IDF-Gewichten

tfidf_scores = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

top_tfidf = sorted(
    zip(tfidf_feature_names, tfidf_scores),
    key=lambda x: x[1],
    reverse=True
)[:30]

top_tfidf

[('credit', 1278.820956253964),
 ('account', 1123.379766396849),
 ('report', 957.1446900259878),
 ('information', 821.2517317780888),
 ('reporting', 803.3644078316764),
 ('consumer', 770.6119152316838),
 ('accounts', 639.2348335424094),
 ('section', 622.7838580502697),
 ('debt', 460.9782693416673),
 ('states', 432.887082025825),
 ('payment', 422.3872547607164),
 ('balance', 404.1573931030927),
 ('company', 395.6370789672366),
 ('agency', 389.96529819826605),
 ('items', 385.5577209379068),
 ('card', 358.35859136434357),
 ('bank', 355.08195258880744),
 ('late', 352.0115806878646),
 ('did', 348.8749965197285),
 ('remove', 348.69187136752885),
 ('inaccurate', 340.4278735182377),
 ('identity', 335.93009593344823),
 ('received', 325.7754331523565),
 ('sent', 323.89065122054325),
 ('loan', 320.129599676933),
 ('theft', 318.09499998607635),
 ('bureaus', 318.0426153759742),
 ('number', 308.17299462795415),
 ('letter', 297.9936173620988),
 ('payments', 294.1480331724286)]

In der letzten Zeile tauchte nach dem ersten Lauf mit „xxxxxxxx“ eine X-Kombination mit mehr als vier Zeichen auf. Deshalb wurde der Code bei beiden Verfahren nochmals angepasst, sodass neben den bereits ausgeschlossenen Zahlen nun auch X-Kombinationen beliebiger Länge ausgeschlossen werden.

## 7. Semantische Analyse

### 7.1 LDA

Zur Themenextraktion wird zunächst LDA auf die bereinigte Bag-of-Words-Matrix angewendet. Als Ausgangswert werden 10 Themen gewählt. Die Anzahl dient zunächst nur als Startwert und soll anschließend anhand der Ergebnisse und des Coherence Scores überprüft werden.

In [39]:
from sklearn.decomposition import LatentDirichletAllocation

# LDA zunächst mit 10 Themen
lda = LatentDirichletAllocation(
    n_components=10,
    random_state=42
)

lda.fit(bow_matrix_clean)

,"random_state random_state: int, RandomState instance or None, default=NonePass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",10
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0


In [40]:
# Wichtigste Wörter der 10 LDA-Themen

feature_names = bow_vectorizer_clean.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    top_words = topic.argsort()[-10:][::-1]
    words = [feature_names[i] for i in top_words]
    print(f"Thema {topic_idx + 1}: {', '.join(words)}")

Thema 1: credit, account, identity, report, fraud, police, accounts, fraudulent, theft, paypal
Thema 2: credit, report, information, theft, reporting, identity, accounts, balance, items, bureaus
Thema 3: consumer, reporting, section, account, information, agency, states, credit, report, furnish
Thema 4: credit, report, information, accounts, reporting, request, balance, matter, items, reports
Thema 5: credit, report, inquiry, information, equifax, accounts, account, date, remove, inquiries
Thema 6: account, bank, told, money, called, card, received, did, said, phone
Thema 7: wells, fargo, bank, claim, court, mortgage, loan, complaint, property, foreclosure
Thema 8: debt, information, consumer, credit, collection, reporting, violation, report, company, provide
Thema 9: account, credit, payment, late, card, balance, payments, paid, charge, time
Thema 10: loan, mortgage, payment, told, payments, time, company, received, pay, did


Bei der Betrachtung der zehn Themen fällt auf, dass sich einige Themen inhaltlich stark überschneiden. Daher wird vermutet, dass eine geringere Anzahl an Themen zu einer klareren Trennung führen könnte. Dies soll im nächsten Schritt mithilfe des Coherence Scores überprüft werden. Da scikit-learn hierfür keine entsprechende Funktion bereitstellt, wird für die Berechnung zusätzlich die Bibliothek Gensim verwendet.

Um zu überprüfen, welche Anzahl an LDA-Themen am sinnvollsten ist, werden anschließend mehrere LDA-Modelle mit unterschiedlichen Anzahlen an Themen berechnet und mithilfe des Coherence Scores miteinander verglichen. Ein höherer c_v-Wert spricht dabei für einen stärkeren inhaltlichen Zusammenhang der Wörter innerhalb der gefundenen Themen.

Für die Berechnung wird die Bibliothek Gensim verwendet. Das bisherige LDA-Modell wurde jedoch mit scikit-learn erstellt. Gensim benötigt für die Berechnung des c_v-Scores zusätzlich die Texte in tokenisierter Form sowie ein eigenes Wörterbuch. Deshalb werden die vorhandenen Texte zunächst mit denselben Einstellungen wie beim CountVectorizer in einzelne Wörter zerlegt und daraus ein Gensim-Dictionary erstellt. Dabei werden noch keine neuen Themen berechnet oder verändert. Dieser Schritt dient lediglich dazu, die vorhandenen Daten für die anschließende Berechnung des Coherence Scores vorzubereiten.

In [43]:
from gensim.corpora import Dictionary

# Texte mit denselben Regeln wie beim CountVectorizer tokenisieren
analyzer = bow_vectorizer_clean.build_analyzer()
tokenized_texts = [analyzer(text) for text in sample["narrative"]]

# Gensim-Wörterbuch für die Berechnung des c_v-Coherence Scores
dictionary = Dictionary(tokenized_texts)

Da sich bei zehn Topics mehrere Themen überschneiden, wird geprüft, ob eine andere Anzahl an Topics zu einer besseren thematischen Struktur führt. Dazu werden verschiedene Topic-Anzahlen berechnet und anhand des c_v-Coherence Scores miteinander verglichen.

In [45]:
from sklearn.decomposition import LatentDirichletAllocation
from gensim.models import CoherenceModel

coherence_scores = []

for n_topics in range(1, 21):

    # LDA für die jeweilige Anzahl an Themen neu berechnen
    lda_test = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42
    )
    lda_test.fit(bow_matrix_clean)

    # Die 10 wichtigsten Wörter jedes Themas bestimmen
    topics = []
    for topic in lda_test.components_:
        top_indices = topic.argsort()[-10:][::-1]
        topics.append(
            [bow_vectorizer_clean.get_feature_names_out()[i]
             for i in top_indices]
        )

    # c_v-Coherence Score berechnen
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence="c_v"
    )

    score = coherence_model.get_coherence()
    coherence_scores.append(score)

    print(f"{n_topics} Themen: {score:.4f}")

1 Themen: 0.5379
2 Themen: 0.4046
3 Themen: 0.5484
4 Themen: 0.5320
5 Themen: 0.5886
6 Themen: 0.5327
7 Themen: 0.5546
8 Themen: 0.5555
9 Themen: 0.5842
10 Themen: 0.5859
11 Themen: 0.5958
12 Themen: 0.6133
13 Themen: 0.5949
14 Themen: 0.5733
15 Themen: 0.5558
16 Themen: 0.5574
17 Themen: 0.5713
18 Themen: 0.5644
19 Themen: 0.5695
20 Themen: 0.5670


Der höchste c_v-Coherence Score wird bei 12 Topics mit 0,6133 erreicht. Im nächsten Schritt wird daher geprüft, ob die zwölf Themen auch inhaltlich eine sinnvolle und klarere Trennung ergeben.

In [47]:
from sklearn.decomposition import LatentDirichletAllocation

lda_12 = LatentDirichletAllocation(
    n_components=12,
    random_state=42
)

lda_12.fit(bow_matrix_clean)

# 10 wichtigste Wörter der 12 Topics ausgeben
feature_names = bow_vectorizer_clean.get_feature_names_out()

for topic_idx, topic in enumerate(lda_12.components_):
    top_words = topic.argsort()[-10:][::-1]
    words = [feature_names[i] for i in top_words]
    print(f"Thema {topic_idx + 1}: {', '.join(words)}")

Thema 1: credit, information, report, score, fraud, police, social, personal, help, security
Thema 2: credit, report, information, reporting, accounts, bureaus, items, fcra, balance, remove
Thema 3: consumer, information, reporting, report, agency, section, person, shall, inaccurate, states
Thema 4: credit, report, information, accounts, balance, reporting, matter, items, request, financial
Thema 5: credit, report, inquiry, information, account, date, equifax, dispute, remove, inquiries
Thema 6: account, bank, told, money, card, called, received, did, said, phone
Thema 7: bank, wells, fargo, court, claim, attorney, complaint, filed, mortgage, foreclosure
Thema 8: debt, collection, credit, consumer, information, violation, company, provide, reporting, act
Thema 9: account, payment, credit, late, payments, balance, card, paid, time, charge
Thema 10: loan, mortgage, told, payment, payments, time, received, company, pay, did
Thema 11: account, section, reporting, states, consumer, credit, 

**Anmerkung:** Bei der Betrachtung der LDA-Ergebnisse fällt auf, dass einzelne Wörter in mehreren Topics vorkommen. Dies bedeutet jedoch nicht zwangsläufig, dass sich auch die Inhalte der Topics überschneiden, da dasselbe Wort in unterschiedlichen thematischen Zusammenhängen auftreten kann.

Topic 1: Kreditbetrug und persönliche Daten  
Topic 2: Fehlerhafte Kreditauskünfte und deren Korrektur  
Topic 3: Meldung von Kreditinformationen und Verbraucherrechte  
Topic 4: Kreditauskünfte und Kontoinformationen  
Topic 5: Kreditanfragen und deren Entfernung  
Topic 6: Probleme mit Bankkonten und Karten  
Topic 7: Hypothekenprobleme und Zwangsvollstreckung bei Wells Fargo  
Topic 8: Inkasso und Meldung von Schulden  
Topic 9: Verspätete Zahlungen und Kreditkartenbelastungen  
Topic 10: Zahlungsprobleme bei Darlehen und Hypotheken  
Topic 11: Kreditauskunft und Verbraucherrechte  
Topic 12: Identitätsdiebstahl und betrügerische Konten

**Interpretation:** Die LDA-Ergebnisse zeigen mehrere gut erkennbare Themenbereiche, gleichzeitig bestehen zwischen einigen Topics deutliche inhaltliche Überschneidungen. Besonders mehrere Topics rund um Kreditauskünfte, Kreditinformationen und Verbraucherrechte verwenden ähnliche Keywords. Andere Bereiche wie Bankkonten und Karten, Hypotheken und Zwangsvollstreckung, Inkasso oder Identitätsdiebstahl lassen sich deutlicher voneinander unterscheiden. Die Themenbezeichnungen wurden anhand der jeweils wichtigsten Keywords interpretiert.

**Anmerkung:**  Die Benennung der Topics erfolgte anhand der jeweils zehn Wörter mit der höchsten Wahrscheinlichkeit und stellt damit eine inhaltliche Interpretation dar. Für eine weiterführende Validierung könnten zusätzlich Dokumente mit einer besonders hohen Zuordnungswahrscheinlichkeit zum jeweiligen Topic betrachtet werden. Dadurch ließe sich überprüfen, ob die gewählten Themenbezeichnungen auch durch die ursprünglichen Beschwerdetexte gestützt werden.

### 7.2 LSA

Als zweite Methode zur Ermittlung latenter Themenstrukturen wird eine Latent Semantic Analysis (LSA) durchgeführt.

Im Gegensatz zu LDA arbeitet LSA nicht mit Wahrscheinlichkeitsverteilungen. Als Grundlage wird die zuvor erstellte TF-IDF-Matrix verwendet. Mithilfe der Singulärwertzerlegung (SVD) wird diese auf eine kleinere Anzahl latenter Dimensionen reduziert. Wörter, die über die Dokumente hinweg ähnliche Strukturen aufweisen, können dadurch in gemeinsamen Dimensionen zusammengefasst werden.

Anschließend werden die Wörter mit den höchsten Gewichten je Dimension betrachtet und die gefundenen Dimensionen inhaltlich interpretiert.

Für eine direkte Vergleichbarkeit mit den zuvor ermittelten 12 LDA-Topics wird die Anzahl der latenten LSA-Dimensionen ebenfalls auf 12 festgelegt.

In [53]:
from sklearn.decomposition import TruncatedSVD

lsa = TruncatedSVD(
    n_components=12,
    random_state=42
)

X_lsa = lsa.fit_transform(tfidf_matrix)

Um die latenten Dimensionen inhaltlich interpretieren zu können, werden für jede der 12 Dimensionen die zehn Wörter mit den höchsten positiven Gewichten ausgegeben. Diese Wörter zeigen, welche Begriffe die jeweilige Dimension besonders stark prägen.

In [55]:
# Wörter aus dem TF-IDF-Vektorisierer
feature_names = tfidf_vectorizer.get_feature_names_out()

# 10 stärkste Wörter je LSA-Dimension
for i, component in enumerate(lsa.components_):
    top_indices = component.argsort()[-10:][::-1]
    top_words = feature_names[top_indices]
    
    print(f"Dimension {i + 1}: {', '.join(top_words)}")

Dimension 1: consumer, section, reporting, account, credit, states, information, report, agency, furnish
Dimension 2: section, states, consumer, furnish, privacy, agency, violated, instructions, rights, accordance
Dimension 3: account, payment, bank, late, card, loan, payments, told, paid, called
Dimension 4: letters, filing, involved, complaint, uploaded, party, continuously, falsely, gotten, misleading
Dimension 5: consumer, information, person, debt, loan, shall, make, relating, told, reasonable
Dimension 6: theft, identity, victim, account, letters, balance, fraud, money, complaint, party
Dimension 7: debt, collection, account, company, validation, alleged, original, collector, contract, owe
Dimension 8: balance, items, information, sections, erroneous, validate, original, owed, creditor, bureaus
Dimension 9: account, number, information, consumer, report, inaccurate, accounts, reports, consent, person
Dimension 10: late, payment, consent, balance, reports, consumer, payments, cred

Die ausgegebenen Wörter stellen jeweils die zehn höchsten positiven Gewichte einer latenten Dimension dar. Anders als bei LDA handelt es sich dabei nicht um Wahrscheinlichkeiten, sondern um Gewichte aus der SVD. Anhand dieser Wörter werden die Dimensionen im Folgenden inhaltlich interpretiert.

Dimension 1: Meldung und Bereitstellung von Kreditauskünften  
Dimension 2: Fehlerhafte Kreditauskünfte und deren Korrektur/Löschung  
Dimension 3: Verspätete Zahlungen und deren Meldung  
Dimension 4: Zahlungsprobleme bei Hypotheken / Kreditanpassungen  
Dimension 5: Scheckeinzahlungen und Verfügbarkeit von Guthaben  
Dimension 6: Kreditbetrug / Identitätsmissbrauch  
Dimension 7: Inkasso und Forderungseinzug  
Dimension 8: Kreditkartenbelastungen und Gebühren  
Dimension 9: Autokredit / Fahrzeugfinanzierung  
Dimension 10: Studentendarlehen und Schuldenerlass  
Dimension 11: Kreditanfragen und unautorisierte Kreditkartenaktivitäten  
Dimension 12: Kreditanfragen in der Kreditauskunft

**Interpretation:** Die LSA-Ergebnisse zeigen mehrere gut erkennbare Themenbereiche. Einzelne Dimensionen, beispielsweise zu Identitätsdiebstahl und Betrug, Inkasso oder Kreditanfragen, sind relativ klar voneinander abgegrenzt. Gleichzeitig bestehen zwischen einigen Dimensionen inhaltliche Überschneidungen, insbesondere bei Themen rund um Kreditauskünfte, Zahlungen und Verbraucherrechte.

## 8. Vergleich der Verfahren

![Vergleich LDA-LSA](Vergleich_LDA_LSA_v2.png)

**Ergebnisse:** Obwohl LDA und LSA mathematisch unterschiedlich arbeiten, zeigen die Ergebnisse eine überraschend große Überschneidung bei den grundlegenden Themen. Viele zentrale Themenbereiche werden von beiden Verfahren erkannt. Dazu gehören beispielsweise Kreditbetrug und Identitätsmissbrauch, fehlerhafte Kreditauskünfte, Scheckeinzahlungen, Studentendarlehen oder Autokredite.

Die Ergebnisse sind dabei nicht identisch. Teilweise fassen die Verfahren Themen unterschiedlich zusammen oder trennen sie stärker voneinander. So werden beispielsweise Zahlungs- und Hypothekenprobleme bei LDA auf mehrere Topics verteilt, während LSA diese teilweise in einer gemeinsamen Dimension zusammenfasst. LSA unterscheidet dagegen verschiedene Bereiche rund um Kreditauskünfte und Kreditanfragen stärker.

Insgesamt zeigen beide Verfahren damit trotz ihrer unterschiedlichen Vorgehensweise ein sehr ähnliches thematisches Grundgerüst des Datensatzes. Die Unterschiede liegen vor allem darin, wie die einzelnen Themen voneinander abgegrenzt bzw. zusammengefasst werden.


**Anmerkung:** Auf eine Lemmatisierung wurde in der ursprünglichen Analyse zunächst verzichtet, da sich die ermittelten Themen bereits sinnvoll interpretieren ließen. In Phase 3 wurde der Einfluss der Lemmatisierung ergänzend untersucht. Dabei reduzierte sich das Vokabular von 21.184 auf 17.922 Merkmale. Eine bessere thematische Abgrenzung oder Interpretierbarkeit der LDA-Topics und LSA-Dimensionen zeigte sich jedoch nicht. Bei LDA lag der höchste c_v-Coherence-Score bei sieben Topics mit 0,5615 und damit niedriger als in der ursprünglichen Analyse. Die ursprüngliche Entscheidung, für die Hauptanalyse auf eine Lemmatisierung zu verzichten, wurde dadurch bestätigt.